In [25]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision import datasets
import torch.nn.functional as F
train_dataset=datasets.MNIST(root='./data/mnist',train=True,transform=transforms.ToTensor(),download=True)
test_dataset=datasets.MNIST(root='./data/mnist',train=False,transform=transforms.ToTensor(),download=True)

train_loader=DataLoader(dataset=train_dataset,batch_size=32,shuffle=True);
test_loader=DataLoader(dataset=test_dataset,batch_size=32,shuffle=False);

In [26]:
class ResidualBlock(torch.nn.Module):
    def __init__(self,channels):
        super(ResidualBlock,self).__init__()
        self.channels=channels
        self.conv1=torch.nn.Conv2d(channels,channels,kernel_size=3,padding=1)
        self.conv2=torch.nn.Conv2d(channels,channels,kernel_size=3,padding=1)
    def forward(self,x):
        y=F.relu(self.conv1(x))
        y=self.conv2(y)
        return F.relu(x+y)

In [27]:
class Net(torch.nn.Module):
    def __init__(self):
        super(Net,self).__init__()
        self.conv1=torch.nn.Conv2d(1,16,kernel_size=5)
        self.conv2=torch.nn.Conv2d(16,32,kernel_size=5)
        self.mp=torch.nn.MaxPool2d(2)
        
        self.rblock1=ResidualBlock(16)
        self.rblock2=ResidualBlock(32)
        
        self.fc=torch.nn.Linear(512,10)
        
    def forward(self,x):
        in_size=x.size(0)
        x=self.mp(F.relu(self.conv1(x)))
        x=self.rblock1(x)
        x=self.mp(F.relu(self.conv2(x)))
        x=self.rblock2(x)
        x=x.view(in_size,-1)
        x=self.fc(x)
        return x
model=Net()

In [28]:
criterion=torch.nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.01,momentum=0.5)
def train(epoch):
    running_loss=0.0
    for batch_idx,(inputs,target) in enumerate(train_loader,0):
        optimizer.zero_grad()
        
        outputs=model(inputs)
        loss=criterion(outputs,target)
        loss.backward()
        optimizer.step()
        
        running_loss+=loss.item()
        if batch_idx %200==0:
            print(f'[{epoch+1} {batch_idx+1}] loss:{running_loss/2000:.3f}')
            running_loss=0.0
        

In [29]:
def test():
    correct=0
    total=0
    with torch.no_grad():
        for data in test_loader:
            inputs,target=data
            outputs=model(inputs)
            _,predicted=torch.max(outputs.data,dim=1)
            total+=target.size(0)
            correct+= (predicted==target).sum().item()
        print("Accuracy: %d %% [%d/%d]" % (100*correct/total,correct,total))
        

In [ ]:
if __name__=="__main__":
    for epoch in range(20):
        train(epoch)
        test()

[1 1] loss:0.000
[1 201] loss:0.000
[1 401] loss:0.000
[1 601] loss:0.000
[1 801] loss:0.000
[1 1001] loss:0.000
[1 1201] loss:0.000
[1 1401] loss:0.000
[1 1601] loss:0.000
[1 1801] loss:0.001
Accuracy: 99 % [9904/10000]
[2 1] loss:0.000
[2 201] loss:0.000
[2 401] loss:0.000
[2 601] loss:0.001
[2 801] loss:0.000
[2 1001] loss:0.000
[2 1201] loss:0.000
[2 1401] loss:0.000
[2 1601] loss:0.000
[2 1801] loss:0.000
Accuracy: 99 % [9907/10000]
[3 1] loss:0.000
[3 201] loss:0.000
[3 401] loss:0.000
[3 601] loss:0.000
[3 801] loss:0.000
[3 1001] loss:0.000
[3 1201] loss:0.000
[3 1401] loss:0.000
[3 1601] loss:0.000
